In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os
import re

import glob
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.impute import KNNImputer

%autosave 120

## ??????: 1_data

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/raw_data'
data1 = pd.read_csv(path_ + r'/AAPRTRAN_20171201_20191231.csv')
data2 = pd.read_csv(path_ + r'/AAPRTRAN_20200101_20220101.csv')
data3 = pd.read_csv(path_ + r'/AAPRTRAN_after_20220101.csv')
data = pd.concat([data1, data2, data3], axis=0)
print(data1.shape)
print(data2.shape)
print(data3.shape)
print(data.shape)
#data

In [ ]:
data = data[data['location'].str.contains('??|??')==True]
data = data[data['district_id'].str.contains('???')==False]
print(data.shape)

In [ ]:
# Duplicated
print(data.duplicated().value_counts())
data[data.duplicated()]

In [ ]:
data = data.rename(columns={data.columns[0]: 'ID'})
print('????:?????????')
print(data.isna().sum()[data.isna().sum()!=0])

In [ ]:
data = data[~data['tsign'].isin(['??', '??'])]
data.shape

In [ ]:
# print('???????')
# print(data.isna().sum()[data.isna().sum()!=0])

In [ ]:
# ???????
data = data.drop(['ncity_z', 'ncity_z_fg', 'land_use', 'land_use_fg', 'abnorm_td', 'abnorm_loc', 'abnorm_prc'], axis=1)
data = data.drop(['coid', 'key3', 'district'], axis=1) #?????
data.shape

In [ ]:
# ?????????
data['mdate'] = pd.to_datetime(data['mdate']).dt.date
data['ann_date'] = pd.to_datetime(data['ann_date']).dt.date
data['build_dd'] = pd.to_datetime(data['build_dd']).dt.date

In [ ]:
# ??? - ?????? ????,????? 0
time_delta = pd.to_timedelta(data['mdate'] - data['build_dd'], errors='coerce')
age = time_delta.dt.days/365
age = age.clip(lower=0)  #???0 ???0

data['room_age']= data['room_age'].fillna(age) #?? age ?????data['room_age']?? NaN

In [ ]:
# ??????????(???)
data = data.drop(['tran_tp_fg','tsign_fg','zoning_fg','mainuse_fg', 'berth_tp_fg', 'matl_fg', 'bstate_fg'], axis=1) #?????
data.shape

In [ ]:
# ??????? ????? ? tran_tp (???) ?????
data.loc[data.index[data['rmk'].str.contains('??')==True], 'tran_tp'] = '???'

# ??????? ????? ? tran_tp (???) ?????
data.loc[data.index[data['rmk'].str.contains('??')==True], 'tran_tp'] = '???'

In [ ]:
# ? berth_prc(????)>0? berth_area(?????)? 0 ??, ?? berth_area ?? NA (??????0...)
data.loc[(data['berth_prc'] > 0) & (data['berth_area'] == 0), 'berth_area'] = np.nan

In [ ]:
# ???????????
# layout = data['room'].astype(str) + '?' + data['hall'].astype(str) + '?' + data['health'].astype(str) + '?'
# data['layout'] = layout

In [ ]:
print('???????')
print(data.isna().sum()[data.isna().sum()!=0])

In [ ]:
# ????
path_clean = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data.to_csv(path_clean + r'/1_data.csv', sep=',', encoding='utf-8', index=False)

## ??????: 2_data

In [ ]:
path_clean = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data = pd.read_csv(path_clean+ r'/1_data.csv')
print(data.shape)

In [ ]:
# ????????????????
def chinese_to_arabic(cn_num):
    chinese_numbers = {
        '?': 1, '?': 2, '?': 2, '?': 3, '?': 4, '?': 5,
        '?': 6, '?': 7, '?': 8, '?': 9, '?': 10
    }
    total = 0
    r = 1  # ????:?????...

    for i in range(len(cn_num) - 1, -1, -1):
        val = chinese_numbers.get(cn_num[i])
        if val is not None:
            if val == 10:
                if i == 0:
                    total += 10
                else:
                    r = 10
            else:
                total += val * r
                r *= 10
    return total

# ?????????
def parse_floor(floor):
    if pd.isna(floor):
        return [np.nan, np.nan]
    # ???????????????
    if floor == '?':
        return[0, np.nan]
    floors = re.findall(r'[???????????]+', floor)
    floors = [chinese_to_arabic(f) for f in floors]
    if len(floors) == 0:
        return [np.nan, np.nan]
    return [min(floors), max(floors)]

In [ ]:
floors = data['tran_floor'].tolist()

parsed_floors = [parse_floor(floor) for floor in floors]
floor_df = pd.DataFrame(parsed_floors, columns=['????', '????'])

data = pd.concat([data, floor_df], axis=1)
data.loc[data['????'] == 0, '????'] = data['tot_floor']
data = data.drop(["tran_floor", "tran_up_ll", "tran_single"], axis=1)
data.shape

In [ ]:
# build_dd????, compart???????, ann_date ??? ?
data = data.drop(['build_dd', 'compart', 'ann_date'], axis=1)
data.shape

In [ ]:
# ??????:?
data = data[data['rmk'].str.contains('????')!=True]
print(data.shape)

# ???????:?
data = data[data['tsign'] != '??']
print(data.shape)

# ????: ????????0 ?
data = data[data['bstate']!='??']
data = data[data['bstate']!='??']
print(data.shape)

# ????: ?? ?? ??? ? ??? ??
data.loc[~data['mainuse'].isin(['??', '???', '???', '???', '???']), 'mainuse'] = '??'
print(data.shape)

In [ ]:
print('???????')
print(data.isna().sum()[data.isna().sum()!=0])

In [ ]:
# ????
path_clean = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data.to_csv(path_clean + r'/2_data.csv', sep=',', encoding='utf-8', index=False)

### ??????: 3_data


In [ ]:
path_clean = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data = pd.read_csv(path_clean+ r'/2_data.csv')
print(data.shape)

In [ ]:
# ?? rmk tot_prc
data = data.drop(['rmk', 'tot_prc'], axis=1)
data.shape

In [ ]:
data.columns = ['ID', '???', '????', '???', '????', '??/??', '??(Y/N)',
       '??', '??', '??', '??', '??', '??????', '????????',
       '???', '????(Y/N)', '????', '????', '????', '??(?)',
       '??????', '??-??', '??-??', '??-???', '????(Y/N)', '????',
       '?????(?)', '??(?/?)', '	????(?)', '????', '????']

In [ ]:
print('???????')
print(data.isna().sum()[data.isna().sum()!=0])

In [ ]:
# ????
data.to_csv(path_clean + r'/3_data.csv', sep=',', encoding='utf-8', index=False)

In [ ]:
# ???? 7532 ????

### ???????

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data6 = pd.read_csv(path_ + r'/6_data.csv')


In [ ]:
data6['??(?/?)'].describe()

In [ ]:
tryy = data6[data6['??(?/?)']==0]
tryy.to_csv(path_ + r'/tryy.csv', sep=',', encoding='utf-8', index=False)

### 7_data
- knnimputer ????
- ????????
- ??????
- ??????:oh, homals


In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data6 = pd.read_csv(path_ + r'/6_data.csv')
data6.shape

In [ ]:
# ????????
# ??????
data6 = data6.drop(['??/??', '????'], axis=1)
data6.shape

In [ ]:
print(data6.isna().sum()[data6.isna().sum()!=0])

In [ ]:
# ????????Unix??(1970?1?1?)?????
data6['???'] = data6['???'].apply(lambda x: pd.to_datetime(x).timestamp())
data6.shape

In [ ]:
# ??
data6_con = data6._get_numeric_data()
print(data6_con.shape)
print(data6_con.columns)

# ??
data6_ca = data6.loc[:, data6.columns.difference(data6_con.columns)]
print(data6_ca.shape)
print(data6_ca.columns)

In [ ]:
data6_con_ca = pd.concat([data6_con, data6_ca], axis = 1)
data6_con_ca.shape

In [ ]:
# ?? KNNImputer
# imputer = KNNImputer(n_neighbors=10, weights = 'distance')
# data6_knnimpute = imputer.fit_transform(np.array(data6_con_ca))
# data6_final = pd.DataFrame(data6_knnimpute, columns=o)

In [ ]:
# data6_ca

In [ ]:
# ??????oh
data6_ca_encoded = pd.get_dummies(data6_ca, columns=data6_ca.columns)
print('one hot encoding', data6_ca_encoded.shape)

# ??????? ?r??homals
# data6_ca.to_csv(path_ + r'/data6_ca.csv', sep=',', encoding='utf-8', index=False)

In [ ]:
# r ??homals
# homals = pd.read_csv(path_ + r'/homals.csv')
# print('homals', homals.shape)

In [ ]:
# ????????????

# oh
data6_oh = pd.concat([data6_con, data6_ca_encoded], axis = 1)
print(data6_oh.shape)

# homals
# data6_homals = pd.concat([data6_con, homals], axis = 1)
# print(data6_homals.shape)

In [ ]:
o = data6_oh.columns
# h = data6_homals.columns

In [ ]:
print(data6_oh.isna().sum()[data6_oh.isna().sum()!=0])

In [ ]:
# knn imputer

# oh
imputer = KNNImputer(n_neighbors=10, weights = 'distance')
oh_knnimpute = imputer.fit_transform(np.array(data6_oh))

# homals
# imputer = KNNImputer(n_neighbors=10, weights = 'distance')
# homals_knnimpute = imputer.fit_transform(np.array(data6_homals))

In [ ]:
# data_homals = pd.DataFrame(homals_knnimpute, columns=h)

In [ ]:
print(data_oh.shape)
# print(data_homals.shape)

In [ ]:
data_oh.to_csv(path_ + r'/7_data_oh.csv', sep=',', encoding='utf-8', index=False)
# data_homals.to_csv(path_ + r'/7_data_homals.csv', sep=',', encoding='utf-8', index=False)

### 11_data
?????0

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data10 = pd.read_csv(path_ + r'/10_data.csv')

In [ ]:
data10.head()

In [ ]:
data11 = data10[data10['??(?/?)']!=0]

In [ ]:
data11.shape

In [ ]:
data11.to_csv(path_ + r'/11_data.csv', sep=',', encoding='utf-8', index=False)

### 12_data ???? train test

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data8 = pd.read_csv(path_ + r'/8_data.csv')

In [ ]:
data8_11 = data8[data8['??(?/?)']!=0]

In [ ]:
new = pd.to_datetime(data8_11['???'], unit='s')[data8_11.shape[0]-33142:]
new.to_csv(path_ + r'/11_data_date.csv', sep=',', encoding='utf-8', index=False)
new

In [ ]:
new[:-6603]

In [ ]:
new[-6603:]

In [ ]:
print(26539/33142)
print(6603/33142)

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data11 = pd.read_csv(path_ + r'/11_data.csv')
data11.shape

In [ ]:
data12_train = data11.iloc[:26539, ] # 2022-06-15??
data12_test = data11.iloc[26539:, ]

data12_train.to_csv(path_ + r'/12_data_train.csv', sep=',', encoding='utf-8', index=False)
data12_test.to_csv(path_ + r'/12_data_test.csv', sep=',', encoding='utf-8', index=False)

In [ ]:
path_ = r'/Users/liang/Documents/NCKU_DS/????/????/clean_data'
data12_train = pd.read_csv(path_ + r'/12_data_train.csv')
data12_test = pd.read_csv(path_ + r'/12_data_test.csv')

print(data12_train.shape)
print(data12_test.shape)

In [ ]:
print(data12_train.isna().sum())#[data12_train.isna().sum()!=0])
print(data12_test.isna().sum())#[data12_train.isna().sum()!=0])